In [69]:
import pandas as pd

def create_initial_df():

    df = pd.DataFrame(columns=['code'])
    print("创建了初始DataFrame，包含code列")
    return df
def add_column_to_df(df, column_name, code, default_value):
    
    df = df.copy()  # 避免修改原DataFrame
    
    # 检查列是否存在
    if column_name in df.columns:
        mask = df['code'] == code
        if mask.any():
            df.loc[mask, column_name] = default_value
        else:
            print(f"未找到code='{code}'的行")
    else:
        # 添加新列，默认值为空或NaN
        df[column_name] = pd.NA
        # 为特定code设置值
        mask = df['code'] == code
        if mask.any():
            df.loc[mask, column_name] = default_value
        else:
            print(f"未找到code='{code}'的行")
    
    return df

In [94]:
def calculate_interval(meet_time,leave_time,results):
    interval=leave_time-meet_time
    for r in results:
        if leave_time>r['time_values']:
            leave_time=leave_time-r['gap']
        else:
             break
    return interval
def calculate_barrier(meet_data,leave_data,df,results,code):
    num_barrier=len(meet_data[0])
    for i in range(num_barrier):
        #print('i:'+str(i))
        interval=0
        for j in range(16):
            #print('j:'+str(j))
            #print(meet_data[j][i])
            #print(leave_data[j][i])
            interval=calculate_interval(meet_data[j][i],leave_data[j][i],results)+interval;
        df=add_column_to_df(df, "barrier_"+str(i), code, interval//16);
    return df
            
        
    

In [103]:
import re
def extract_time_from_line(line):
    pattern = r'\*+time:(\d+)\*+'
    
    match = re.search(pattern, line, re.IGNORECASE)  # 忽略大小写
    if match:
        return int(match.group(1))  # 返回第一个捕获组（数字部分）
    return None
def extract_barrier_from_line(line):
   if 'barrier' in line.lower():
        return True
   else:
       return False
def extract_time_from_txt_file(filename):
    results=[]
    flag=False
    counter=0
    p_time=0
    time_value_temp=0
    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as file:
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                flag=extract_barrier_from_line(line)
                time_value = extract_time_from_line(line)
                
                if time_value:
                    time_value_temp=time_value
                    counter=counter+1
                    if(counter==3):
                        result = {
                            'time_values': int,
                            'gap': int,
                        }
                        result['gap']=time_value-p_time
                        result['time_values']=time_value
                        results.append(result)
                    elif(counter<3):
                        p_time=time_value
                    else:
                        assert(0)
                if flag==True:
                    flag=False
                    counter=0;
    except Exception as e:
        print(f"读取文件 {filename} 时出错: {e}")
    time_value_real=time_value_temp
    for r in results:
        if time_value_temp>=r['time_values']:
            time_value_real=time_value_real-r['gap']
        else:
            assert(0)
    return results,time_value_real


In [101]:
from pathlib import Path

def extract_thread_and_time(line):
    
    pattern = r'fthread\s+(\d+)\s+(meet|leave)\s+barrier(\d+)'
    
    match = re.search(pattern, line)
    if match:
        return {
            'thread_id': int(match.group(1)),      # 线程号
            'action': match.group(2),              # 动作 (meet/leave)
            'time': int(match.group(3)),           # 时间
        }
    
    return None
def find_syn_file(filename):
    meet_data = {}
    for i in range(16):
        meet_data[i] = []
    leave_data = {}
    for i in range(16):
        leave_data[i] = []

 
    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as file:
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                res=extract_thread_and_time(line)
                if res is not None:
                    if res['action']=='meet':
                        meet_data[res['thread_id']].append(res['time'])
                    elif res['action']=='leave':
                        leave_data[res['thread_id']].append(res['time'])
                    else:
                        assert(0)
    except Exception as e:
        print(f"读取文件 {filename} 时出错: {e}")
    
    return meet_data,leave_data


In [104]:
import os
import glob
from pathlib import Path
import csv
import pandas as pd
import re
#
"""使用os.listdir遍历当前目录中包含'txt'的文件"""
df=create_initial_df()
for filename in os.listdir("."):
    if 'txt' in filename:
        numbers = re.findall(r'\d+', filename)
        new_row = pd.DataFrame({'code': [str(numbers[0])]})
        df = pd.concat([df, new_row], ignore_index=True)
        results,time_value = extract_time_from_txt_file(filename)
        df=add_column_to_df(df, "total_cycle", str(numbers[0]), time_value);
        meet_data,leave_data=find_syn_file(filename)
        
        df= calculate_barrier(meet_data,leave_data,df,results,str(numbers[0])); 
        print(filename+" is finsihed")     
df.to_csv('result.csv', index=False, encoding='utf-8')

创建了初始DataFrame，包含code列
22.txt is finsihed
15.txt is finsihed
23.txt is finsihed
11.txt is finsihed
29.txt is finsihed
12.txt is finsihed
17.txt is finsihed
21.txt is finsihed
19.txt is finsihed
26.txt is finsihed
